# Colab Training LoRA đầy đủ: Poster + Logo 2D + Logo 3D

Notebook này huấn luyện **3 LoRA riêng biệt**:
1. `poster_lora`
2. `logo2d_lora`
3. `logo3d_lora`

Sau khi train xong, bạn sẽ có các thư mục output có thể tải về local để dùng trong backend.

---

## Yêu cầu trước khi chạy
- Runtime: **GPU** (T4/L4/A100 đều được)
- Dataset đã upload lên Google Drive theo cấu trúc:

```
MyDrive/AI_GEN/dataset/
  POSTER/
    POSTER (1).png
    POSTER (1).txt
    ...
  LOGO/
    LOGO 2D/
      LOGO 2D (1).png
      LOGO 2D (1).txt
      ...
    LOGO 3D/
      LOGO 3D (1).png
      LOGO 3D (1).txt
      ...
```


In [ ]:
# Bước 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Bước 2: Cài dependencies
!pip -q install --upgrade pip
!pip -q install diffusers==0.31.0 transformers==4.46.3 accelerate==1.1.1 peft==0.13.2 safetensors==0.4.5 sentencepiece==0.2.0
!pip -q install xformers --index-url https://download.pytorch.org/whl/cu121 || true

In [ ]:
# Bước 3: Clone repo hoặc dùng code trực tiếp trong Drive
import os
REPO_DIR = '/content/AI_GEN'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/your-org/AI_GEN.git /content/AI_GEN
else:
    print('Repo already exists:', REPO_DIR)

## Bước 4: Khai báo đường dẫn dữ liệu

Nếu bạn để dataset ở chỗ khác, sửa các biến bên dưới cho đúng.

In [ ]:
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/AI_GEN')
DATASET_ROOT = DRIVE_ROOT / 'dataset'
WORK_ROOT = Path('/content/work_datasets')
OUTPUT_ROOT = DRIVE_ROOT / 'lora_outputs'

POSTER_SRC = DATASET_ROOT / 'POSTER'
LOGO2D_SRC = DATASET_ROOT / 'LOGO' / 'LOGO 2D'
LOGO3D_SRC = DATASET_ROOT / 'LOGO' / 'LOGO 3D'

for p in [POSTER_SRC, LOGO2D_SRC, LOGO3D_SRC]:
    print(p, 'exists=', p.exists())

WORK_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

## Bước 5: Chuẩn hóa dữ liệu sang cấu trúc `images/` + `captions/`

Script train hiện có yêu cầu mỗi dataset con theo dạng:
```
dataset_name/
  images/*.png|jpg|jpeg
  captions/*.txt
```

In [ ]:
import shutil

IMG_EXTS = {'.png', '.jpg', '.jpeg', '.webp'}

def prepare_dataset(src_dir: Path, out_dir: Path):
    images_dir = out_dir / 'images'
    caps_dir = out_dir / 'captions'
    if out_dir.exists():
        shutil.rmtree(out_dir)
    images_dir.mkdir(parents=True, exist_ok=True)
    caps_dir.mkdir(parents=True, exist_ok=True)

    count = 0
    missing_caps = 0
    for f in sorted(src_dir.iterdir()):
        if f.suffix.lower() not in IMG_EXTS:
            continue
        stem = f.stem
        cap = src_dir / f'{stem}.txt'

        shutil.copy2(f, images_dir / f.name)
        if cap.exists():
            shutil.copy2(cap, caps_dir / cap.name)
        else:
            (caps_dir / f'{stem}.txt').write_text(stem.replace('_', ' '), encoding='utf-8')
            missing_caps += 1
        count += 1

    print(f'Prepared {out_dir.name}: images={count}, missing_caps={missing_caps}')
    return out_dir

POSTER_DS = prepare_dataset(POSTER_SRC, WORK_ROOT / 'poster')
LOGO2D_DS = prepare_dataset(LOGO2D_SRC, WORK_ROOT / 'logo2d')
LOGO3D_DS = prepare_dataset(LOGO3D_SRC, WORK_ROOT / 'logo3d')

## Bước 6: Kiểm tra nhanh dữ liệu
- Mục tiêu: đảm bảo ảnh/caption khớp nhau trước khi train.

In [ ]:
from random import sample

def inspect_dataset(ds_root: Path, n=3):
    images = sorted((ds_root / 'images').glob('*'))
    print('Dataset:', ds_root.name, 'total_images=', len(images))
    pick = sample(images, min(n, len(images)))
    for img in pick:
        cap = ds_root / 'captions' / f'{img.stem}.txt'
        txt = cap.read_text(encoding='utf-8')[:220].replace('\n', ' ') if cap.exists() else 'NO CAPTION'
        print('-', img.name, '=>', txt)

inspect_dataset(POSTER_DS)
inspect_dataset(LOGO2D_DS)
inspect_dataset(LOGO3D_DS)

## Bước 7: Cấu hình training (best-practice cho dữ liệu của bạn)

### Vì sao train 3 LoRA tách biệt?
- Poster và logo có style khác nhau rõ rệt
- Tách LoRA giúp inference dễ kiểm soát và chất lượng ổn định hơn

### Hyperparameters khuyến nghị
- `model_name`: `stabilityai/sdxl-turbo` (nhẹ, phù hợp T4/L4)
- `rank=16`, `alpha=32` (giữ chi tiết style tốt hơn rank 8)
- `lr=1e-4`
- `resolution=512` (an toàn VRAM)
- `batch_size=1`, `grad_accum=8`
- `epochs`: poster 10, logo2d 8, logo3d 8 (có thể tăng nếu data lớn)

In [ ]:
import subprocess
import sys
from pathlib import Path

TRAIN_SCRIPT = Path('/content/AI_GEN/training/lora_trainer/train_lora.py')
assert TRAIN_SCRIPT.exists(), f'Missing train script: {TRAIN_SCRIPT}'

COMMON = {
    'model_name': 'stabilityai/sdxl-turbo',
    'rank': 16,
    'alpha': 32,
    'lr': 1e-4,
    'batch_size': 1,
    'grad_accum': 8,
    'resolution': 512,
    'save_steps': 100,
}

TRAIN_JOBS = [
    {
        'name': 'poster_lora',
        'dataset': str(POSTER_DS),
        'epochs': 10,
        'validation_prompt': 'cinematic movie poster, bold typography, tropical adventure mood'
    },
    {
        'name': 'logo2d_lora',
        'dataset': str(LOGO2D_DS),
        'epochs': 8,
        'validation_prompt': 'minimal flat 2d logo, vector-like clean shape, professional branding'
    },
    {
        'name': 'logo3d_lora',
        'dataset': str(LOGO3D_DS),
        'epochs': 8,
        'validation_prompt': 'luxury 3d logo render, metallic lighting, premium brand identity'
    },
]

TRAIN_JOBS

## Bước 8: Train toàn bộ (chạy tuần tự 3 job)
- Thời gian phụ thuộc GPU + số lượng ảnh
- Nếu lỗi VRAM: giảm `rank` còn 8 hoặc giảm `resolution` còn 384

In [ ]:
def run_job(job):
    out = OUTPUT_ROOT / job['name']
    out.mkdir(parents=True, exist_ok=True)

    cmd = [
        sys.executable, str(TRAIN_SCRIPT),
        '--model_name', COMMON['model_name'],
        '--dataset', job['dataset'],
        '--output', str(out),
        '--rank', str(COMMON['rank']),
        '--alpha', str(COMMON['alpha']),
        '--lr', str(COMMON['lr']),
        '--epochs', str(job['epochs']),
        '--batch_size', str(COMMON['batch_size']),
        '--grad_accum', str(COMMON['grad_accum']),
        '--resolution', str(COMMON['resolution']),
        '--save_steps', str(COMMON['save_steps']),
        '--validation_prompt', job['validation_prompt'],
    ]

    print('\n===== RUN JOB:', job['name'], '=====')
    print('Output:', out)
    subprocess.run(cmd, check=True)

for job in TRAIN_JOBS:
    run_job(job)

print('All training jobs completed.')

## Bước 9: Kiểm tra output cuối
Mỗi job sẽ có thư mục `final/unet_lora` chứa adapter để dùng lại local.

In [ ]:
for job in TRAIN_JOBS:
    final_dir = OUTPUT_ROOT / job['name'] / 'final' / 'unet_lora'
    print('\n', job['name'], '=>', final_dir)
    if final_dir.exists():
        for f in sorted(final_dir.iterdir()):
            print(' -', f.name)
    else:
        print(' - MISSING final output')

## Bước 10: Nén file để tải về local
Bạn có thể zip từng LoRA rồi tải về máy hoặc giữ trên Drive.

In [ ]:
import shutil

for job in TRAIN_JOBS:
    src = OUTPUT_ROOT / job['name'] / 'final' / 'unet_lora'
    zip_base = OUTPUT_ROOT / f"{job['name']}_unet_lora"
    if src.exists():
        shutil.make_archive(str(zip_base), 'zip', root_dir=str(src))
        print('Created:', str(zip_base) + '.zip')
    else:
        print('Skip zip (missing):', src)

## Cách dùng LoRA đã train trong local project

1. Giải nén `*_unet_lora.zip`
2. Copy vào local, ví dụ:
   - `models/lora/poster/unet_lora/`
   - `models/lora/logo2d/unet_lora/`
   - `models/lora/logo3d/unet_lora/`
3. Khi generate, truyền `lora_path` tương ứng hoặc map theo loại task
4. Không nên gộp 3 style vào 1 adapter nếu cần chất lượng tối đa

---

## Mẹo tối ưu chất lượng
- Nếu ảnh output chưa đủ sắc nét: tăng epochs 20-30%
- Nếu bị overfit (ảnh lặp/rụng chi tiết): giảm epochs hoặc lr còn `8e-5`
- Caption nên giữ ngắn gọn + giàu keyword style chính
- Với logo 2D, caption nên nhấn mạnh: `flat`, `vector`, `minimal`, `clean shape`
- Với logo 3D, caption nên nhấn mạnh: `3d`, `metallic`, `lighting`, `depth`, `shadow`